# Liquid Rocket Motor Design Calculator

A first-pass design tool for liquid bipropellant rocket engines.  
All units are **SI (metric)**.

In [ ]:
import sys
sys.path.insert(0, "..")

import numpy as np
import matplotlib.pyplot as plt
from lrd import propellants, nozzle, combustion, injector, tanks, cooling, performance

## 1. Design Requirements

In [ ]:
# --- Design inputs (all metric) ---
THRUST_TARGET = 10_000        # N
CHAMBER_PRESSURE = 2.07e6     # Pa  (~20.7 bar)
PROPELLANT = "N2O/ETHANOL"
AMBIENT_PRESSURE = 101325     # Pa (sea level)
BURN_TIME = 30                # s
CHAMBER_DIAMETER = 0.050      # m (50 mm) — set to None to derive from contraction ratio

print(f"Target thrust:     {THRUST_TARGET:,.0f} N")
print(f"Chamber pressure:  {CHAMBER_PRESSURE/1e5:.1f} bar")
print(f"Propellant:        {PROPELLANT}")
print(f"Burn time:         {BURN_TIME} s")
print(f"Chamber diameter:  {CHAMBER_DIAMETER*1000:.0f} mm" if CHAMBER_DIAMETER else "Chamber diameter:  (from contraction ratio)")

## 2. Propellant Properties

In [ ]:
prop = propellants.get_propellant(PROPELLANT)

print(f"Available propellants: {propellants.list_propellants()}")
print()
for key, val in prop.items():
    print(f"  {key:>15s}: {val}")

## 3. Nozzle Design

In [ ]:
gamma = prop["gamma"]
c_star = prop["c_star"]
pe_pc = AMBIENT_PRESSURE / CHAMBER_PRESSURE

# Expansion ratio and thrust coefficient
eps = nozzle.expansion_ratio_from_pressure(gamma, pe_pc)
cf = nozzle.thrust_coefficient(gamma, eps, pe_pc, pe_pc)
isp = performance.specific_impulse(c_star, cf)
Me = nozzle.exit_mach(gamma, eps)

# Throat sizing
At = THRUST_TARGET / (cf * CHAMBER_PRESSURE)
dt = nozzle.diameter_from_area(At)
Ae = nozzle.exit_area(At, eps)
de = nozzle.diameter_from_area(Ae)

print(f"Expansion ratio:     {eps:.2f}")
print(f"Exit Mach number:    {Me:.2f}")
print(f"Thrust coefficient:  {cf:.3f}")
print(f"Specific impulse:    {isp:.1f} s")
print(f"Throat diameter:     {dt*1000:.1f} mm")
print(f"Exit diameter:       {de*1000:.1f} mm")

In [ ]:
# Nozzle contour plot
x, y = nozzle.nozzle_contour(dt / 2, eps, half_angle_deg=15)

fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(x * 1000, y * 1000, "b-", linewidth=2)
ax.plot(x * 1000, -y * 1000, "b-", linewidth=2)
ax.set_xlabel("Axial distance from throat [mm]")
ax.set_ylabel("Radius [mm]")
ax.set_title("Nozzle Contour (conical, 15° half-angle)")
ax.set_aspect("equal")
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 4. Combustion Chamber Sizing

In [ ]:
CONTRACTION_RATIO = 3.0  # fallback if CHAMBER_DIAMETER is None

l_star = combustion.l_star_typical(PROPELLANT)
Vc = combustion.chamber_volume(At, l_star)
chamber = combustion.chamber_dimensions(Vc, CONTRACTION_RATIO, At, chamber_diameter=CHAMBER_DIAMETER)
CONTRACTION_RATIO = chamber["contraction_ratio"]  # update to actual value
ts = combustion.stay_time(l_star, c_star, CHAMBER_PRESSURE, prop["T_c"], prop["mol_weight"])

print(f"L* (characteristic length): {l_star:.2f} m")
print(f"Chamber volume:    {Vc*1e6:.1f} cm³")
print(f"Chamber diameter:  {chamber['diameter']*1000:.1f} mm")
print(f"Chamber length:    {chamber['length']*1000:.1f} mm")
print(f"Contraction ratio: {CONTRACTION_RATIO:.2f}" +
      (" (from specified diameter)" if CHAMBER_DIAMETER is not None else ""))
print(f"Residence time:    {ts*1000:.2f} ms")

## 5. Injector Sizing

In [ ]:
# Mass flow rates
mdot = performance.mass_flow_rate(THRUST_TARGET, isp)
flows = performance.mixture_ratio_flows(mdot, prop["of_ratio"])

print(f"Total mdot:  {mdot*1000:.1f} g/s")
print(f"Ox mdot:     {flows['mdot_ox']*1000:.1f} g/s")
print(f"Fuel mdot:   {flows['mdot_fuel']*1000:.1f} g/s")

In [ ]:
CD = 0.7
INJ_DP_FRACTION = 0.20  # 20% of Pc
dp_inj = INJ_DP_FRACTION * CHAMBER_PRESSURE

# Ox orifices (2 mm diameter)
d_ox = 0.002  # m
n_ox = injector.orifice_count(flows["mdot_ox"], CD, d_ox, dp_inj, prop["ox_density"])

# Fuel orifices (1.5 mm diameter)
d_fuel = 0.0015  # m
n_fuel = injector.orifice_count(flows["mdot_fuel"], CD, d_fuel, dp_inj, prop["fuel_density"])

check = injector.check_pressure_drop_ratio(dp_inj, CHAMBER_PRESSURE)

print(f"Injector ΔP:       {dp_inj/1e5:.1f} bar")
print(f"Ox orifices:       {n_ox} × {d_ox*1000:.1f} mm dia")
print(f"Fuel orifices:     {n_fuel} × {d_fuel*1000:.1f} mm dia")
print(f"ΔP check:          {check['message']}")

## 6. Feed System & Tanks

In [ ]:
m_ox = flows["mdot_ox"] * BURN_TIME
m_fuel = flows["mdot_fuel"] * BURN_TIME

v_ox = tanks.tank_volume(m_ox, prop["ox_density"])
v_fuel = tanks.tank_volume(m_fuel, prop["fuel_density"])

feed_p = tanks.feed_pressure(CHAMBER_PRESSURE, dp_inj)

# Tank mass estimates (Al 6061-T6, yield ~276 MPa)
AL_YIELD = 276e6  # Pa
ox_tank = tanks.tank_mass_thin_wall(v_ox, feed_p, AL_YIELD)
fuel_tank = tanks.tank_mass_thin_wall(v_fuel, feed_p, AL_YIELD)

print(f"Oxidizer mass:     {m_ox*1000:.0f} g")
print(f"Fuel mass:         {m_fuel*1000:.0f} g")
print(f"Total propellant:  {(m_ox + m_fuel)*1000:.0f} g")
print()
print(f"Ox tank volume:    {v_ox*1000:.2f} L")
print(f"Fuel tank volume:  {v_fuel*1000:.2f} L")
print(f"Feed pressure:     {feed_p/1e5:.1f} bar")
print()
print(f"Ox tank mass:      {ox_tank['mass']*1000:.0f} g  (wall: {ox_tank['wall_thickness']*1000:.2f} mm)")
print(f"Fuel tank mass:    {fuel_tank['mass']*1000:.0f} g  (wall: {fuel_tank['wall_thickness']*1000:.2f} mm)")

## 7. Regenerative Cooling Analysis

In [ ]:
# --- Cooling design inputs ---
WALL_K = 390            # W/(m·K), copper
WALL_THICKNESS = 0.002  # m (2 mm)
N_CHANNELS = 24
CHANNEL_WIDTH = 0.002   # m (2 mm)
CHANNEL_HEIGHT = 0.003  # m (3 mm)
T_COOLANT_IN = 293.0    # K (20°C)

# Ethanol coolant properties
COOLANT_RHO = 789       # kg/m³
COOLANT_CP = 2440       # J/(kg·K)
COOLANT_MU = 1.1e-3     # Pa·s
COOLANT_K = 0.171       # W/(m·K)

# Build full engine contour (chamber + convergent + divergent)
contour = nozzle.full_nozzle_contour(
    dt / 2, eps, CONTRACTION_RATIO,
    chamber_length=chamber["length"],
    chamber_radius=chamber["diameter"] / 2,
)

# Run regen analysis (coolant = all fuel flow)
regen = cooling.regen_cooling_analysis(
    contour["x"], contour["r"], contour["throat_index"],
    CHAMBER_PRESSURE, c_star, prop["T_c"], gamma, prop["mol_weight"],
    dt, N_CHANNELS, CHANNEL_WIDTH, CHANNEL_HEIGHT,
    WALL_THICKNESS, WALL_K,
    flows["mdot_fuel"], COOLANT_RHO, COOLANT_CP, COOLANT_MU, COOLANT_K,
    T_COOLANT_IN,
)

# Key results
peak_idx = int(np.argmax(regen["heat_flux"]))
print(f"Peak heat flux:        {regen['heat_flux'][peak_idx]/1e6:.2f} MW/m²")
print(f"Peak wall temp (gas):  {regen['T_wall_gas'][peak_idx]:.0f} K")
print(f"Coolant outlet temp:   {regen['T_coolant'][0]:.0f} K")
print(f"Coolant ΔT:            {regen['T_coolant'][0] - T_COOLANT_IN:.1f} K")
print(f"Coolant pressure drop: {regen['pressure_drop'][0]/1e5:.2f} bar")
print(f"Coolant velocity:      {regen['coolant_velocity']:.1f} m/s")
print(f"Coolant Reynolds:      {regen['Re_coolant']:.0f}")
print(f"Total heat absorbed:   {regen['total_heat']:.0f} W")

In [ ]:
x_mm = regen["x"] * 1000  # convert to mm

fig, axes = plt.subplots(2, 2, figsize=(14, 9))

# 1. Heat flux profile
ax = axes[0, 0]
ax.plot(x_mm, regen["heat_flux"] / 1e6, "r-", linewidth=2)
ax.set_ylabel("Heat flux [MW/m²]")
ax.set_title("Gas-side Heat Flux")
ax.grid(True, alpha=0.3)

# 2. Temperature profiles
ax = axes[0, 1]
ax.plot(x_mm, regen["T_aw"], "k--", label="T_aw (adiabatic wall)")
ax.plot(x_mm, regen["T_wall_gas"], "r-", linewidth=2, label="T_wall (gas side)")
ax.plot(x_mm, regen["T_wall_coolant"], "b-", linewidth=2, label="T_wall (coolant side)")
ax.plot(x_mm, regen["T_coolant"], "c-", linewidth=2, label="T_coolant")
ax.set_ylabel("Temperature [K]")
ax.set_title("Temperature Profiles")
ax.legend(fontsize=8)
ax.grid(True, alpha=0.3)

# 3. Nozzle contour (for spatial reference)
ax = axes[1, 0]
ax.plot(x_mm, contour["r"] * 1000, "b-", linewidth=2)
ax.plot(x_mm, -contour["r"] * 1000, "b-", linewidth=2)
ax.axvline(x_mm[contour["throat_index"]], color="gray", linestyle="--", alpha=0.5, label="Throat")
ax.set_xlabel("Axial position [mm]")
ax.set_ylabel("Radius [mm]")
ax.set_title("Engine Contour")
ax.set_aspect("equal")
ax.legend(fontsize=8)
ax.grid(True, alpha=0.3)

# 4. Coolant pressure drop
ax = axes[1, 1]
ax.plot(x_mm, regen["pressure_drop"] / 1e5, "g-", linewidth=2)
ax.set_xlabel("Axial position [mm]")
ax.set_ylabel("Pressure drop [bar]")
ax.set_title("Cumulative Coolant Pressure Drop")
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 8. Performance Summary

In [ ]:
summary = performance.engine_summary(
    thrust_target=THRUST_TARGET,
    chamber_pressure=CHAMBER_PRESSURE,
    propellant_key=PROPELLANT,
    ambient_pressure=AMBIENT_PRESSURE,
    burn_time=BURN_TIME,
    contraction_ratio=CONTRACTION_RATIO,
    chamber_diameter=CHAMBER_DIAMETER,
)

print("=" * 50)
print("    ENGINE DESIGN SUMMARY")
print("=" * 50)
print(f"  Propellant:          {summary['propellant']}")
print(f"  Thrust:              {summary['thrust']:,.0f} N")
print(f"  Isp:                 {summary['isp']:.1f} s")
print(f"  Chamber pressure:    {summary['chamber_pressure']/1e5:.1f} bar")
print(f"  Expansion ratio:     {summary['expansion_ratio']:.2f}")
print(f"  C_F:                 {summary['thrust_coefficient']:.3f}")
print(f"  c*:                  {summary['c_star']:.0f} m/s")
print("-" * 50)
print(f"  Throat diameter:     {summary['throat_diameter']*1000:.1f} mm")
print(f"  Exit diameter:       {summary['exit_diameter']*1000:.1f} mm")
print(f"  Chamber diameter:    {summary['chamber_diameter']*1000:.1f} mm")
print(f"  Chamber length:      {summary['chamber_length']*1000:.1f} mm")
print(f"  Contraction ratio:   {summary['contraction_ratio']:.2f}")
print("-" * 50)
print(f"  Mass flow (total):   {summary['mdot_total']*1000:.1f} g/s")
print(f"  Mass flow (ox):      {summary['mdot_ox']*1000:.1f} g/s")
print(f"  Mass flow (fuel):    {summary['mdot_fuel']*1000:.1f} g/s")
print(f"  O/F ratio:           {summary['of_ratio']:.2f}")
print("-" * 50)
print(f"  Burn time:           {summary['burn_time']} s")
print(f"  Ox mass:             {summary['ox_mass']*1000:.0f} g")
print(f"  Fuel mass:           {summary['fuel_mass']*1000:.0f} g")
print(f"  Feed pressure:       {summary['feed_pressure']/1e5:.1f} bar")
print("=" * 50)